### IMPORTS

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key=os.getenv("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", google_api_key=api_key
)

### 1. Define the Schema using Pydanti

In [2]:
from typing import Annotated, Optional, Literal
from pydantic import BaseModel, Field

class Review(BaseModel):
    """Analyze a smartphone review."""
    key_themes: Annotated[
        list[str],
        Field(
            ...,
            description="write down all the themes discussed in the review in a list",
        ),
    ]
    summary: Annotated[str, Field(..., description="A brief summary of the review")]
    sentiment: Annotated[
        Literal["pos", "neg"],
        Field(
            ...,
            description="Return sentiment of the review either negative,positive or neutral",
        ),
    ]
    pros: Annotated[
        Optional[list[str]],
        Field(description="Write down all the pros inside a list", default=None),
    ]
    cons: Annotated[
        Optional[list[str]],
        Field(description="Write down all the cons inside a list", default=None),
    ]
    name: Annotated[
        Optional[str],
        Field(description="Write down the name of the reviewer", default=None),
    ]


### 2. Attach the schema

In [4]:
structured_model = model.with_structured_output(Review)

### 2. 3. Invoke 

In [5]:
result = structured_model.invoke(
    """I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Kristal Shrestha """
)


###  4. Use the structured output (Pydantic object)

In [6]:
print(result)
print(type(result))
print(result.summary)
print(result.sentiment)

print(result.name)

key_themes=['processor performance', 'battery life', 'charging speed', 'S-Pen functionality', 'camera quality', 'zoom capabilities', 'design and ergonomics', 'software bloatware', 'price'] summary='The Samsung Galaxy S24 Ultra is a powerful smartphone with a fast processor, long-lasting battery, and an impressive 200MP camera with good zoom capabilities. However, it is heavy, has some bloatware, and is quite expensive.' sentiment='pos' pros=['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'] cons=['heavy and uncomfortable for one-handed use', 'bloatware in One UI', 'expensive price tag'] name='Kristal Shrestha'
<class '__main__.Review'>
The Samsung Galaxy S24 Ultra is a powerful smartphone with a fast processor, long-lasting battery, and an impressive 200MP camera with good zoom capabilities. However, it is heavy, has some bloatware, a

# Convert to dictionary if needed

In [7]:
result_dict=result.model_dump()
print(result_dict)
print(type(result_dict))
print(result_dict["summary"])
print(result_dict["sentiment"])

print(result_dict["name"])


{'key_themes': ['processor performance', 'battery life', 'charging speed', 'S-Pen functionality', 'camera quality', 'zoom capabilities', 'design and ergonomics', 'software bloatware', 'price'], 'summary': 'The Samsung Galaxy S24 Ultra is a powerful smartphone with a fast processor, long-lasting battery, and an impressive 200MP camera with good zoom capabilities. However, it is heavy, has some bloatware, and is quite expensive.', 'sentiment': 'pos', 'pros': ['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], 'cons': ['heavy and uncomfortable for one-handed use', 'bloatware in One UI', 'expensive price tag'], 'name': 'Kristal Shrestha'}
<class 'dict'>
The Samsung Galaxy S24 Ultra is a powerful smartphone with a fast processor, long-lasting battery, and an impressive 200MP camera with good zoom capabilities. However, it is heavy, has som

### Convert to json directly from pydantic response if needed

In [13]:
import json

# Convert Pydantic to JSON string
result_json = result.model_dump_json()
print(f"JSON string: {result_json}")
print(f"Type: {type(result_json)}")

# Parse JSON string back to dictionary
result_dict = json.loads(result_json)
print(f"\nAccessing as dict:")
print(f"Summary: {result_dict['summary']}")
print(f"Sentiment: {result_dict['sentiment']}")
print(f"Name: {result_dict['name']}")

JSON string: {"key_themes":["processor performance","battery life","charging speed","S-Pen functionality","camera quality","zoom capabilities","design and ergonomics","software bloatware","price"],"summary":"The Samsung Galaxy S24 Ultra is a powerful smartphone with a fast processor, long-lasting battery, and an impressive 200MP camera with good zoom capabilities. However, it is heavy, has some bloatware, and is quite expensive.","sentiment":"pos","pros":["Insanely powerful processor (great for gaming and productivity)","Stunning 200MP camera with incredible zoom capabilities","Long battery life with fast charging","S-Pen support is unique and useful"],"cons":["heavy and uncomfortable for one-handed use","bloatware in One UI","expensive price tag"],"name":"Kristal Shrestha"}
Type: <class 'str'>

Accessing as dict:
Summary: The Samsung Galaxy S24 Ultra is a powerful smartphone with a fast processor, long-lasting battery, and an impressive 200MP camera with good zoom capabilities. Howeve